# IT244 Arena Tools

Trace a single read-only lookup: question → validated arguments → Python tool → evidence → answer. Compare three models from the live Arena text style control leaderboard.

Today: read current evidence, follow the tool trace, and complete the matching 10-question activity. Use Python 3.10+ on CPU; no GPU is needed.

Run the definition cells first, then the live-data cell. Configure the controller only for the LLM demonstration.


## 1. Setup

Data are retrieved during the class run. Nothing here loads an archived benchmark or generates model observations. The small assertions check software behavior; they are not benchmark measurements.

Arena needs network access but no data API key. Install its live file reader once in a notebook cell: `%pip install -q pyarrow`. The source adapter reads the latest published Parquet file into memory. 

For the LLM demonstration, set LLM_MODEL and LLM_API_KEY; optionally set LLM_BASE_URL to the course’s approved Chat Completions compatible gateway. The selected model must support tool calls. Use the platform’s environment/secret setup; save notebooks with outputs cleared.


In [ ]:
import json
import math
import os
from datetime import datetime, timezone
from urllib.request import Request, urlopen
from urllib.parse import urlencode
from urllib.error import HTTPError, URLError

def get_bytes(url, headers=None):
    request = Request(url, headers={"User-Agent": "Teaching-FA26/1.0", **(headers or {})})
    try:
        with urlopen(request, timeout=30) as response:
            return response.read()
    except HTTPError as error:
        # Do not print request headers or API keys.
        raise RuntimeError(f"Source returned HTTP {error.code}; check access or quota.") from None
    except (URLError, TimeoutError):
        raise RuntimeError("Source could not be reached; check network access and rerun.") from None

def get_json(url, headers=None):
    return json.loads(get_bytes(url, headers))

def utc_now():
    return datetime.now(timezone.utc).isoformat(timespec="seconds")

def show(value):
    print(json.dumps(value, indent=2, ensure_ascii=False))

def expect_value_error(function, *args):
    try:
        function(*args)
    except ValueError:
        return
    raise AssertionError("Expected ValueError")

ARENA_SOURCE = "https://huggingface.co/datasets/lmarena-ai/leaderboard-dataset"
# Keep a small, supported category menu. Each request reads the latest split.
ARENA_CATEGORIES = ("overall", "coding", "math")


## 2. Small functions

These ten functions are the exact targets of Q1–Q10 in the activity. The teaching version is complete. Read each function and predict its check before running it.


**Q1 · Clean model names · Easy**

Return stripped, lowercase names; leave out empty names.


In [ ]:
def normalize_names(names):
    return [name.strip().casefold() for name in names if name.strip()]

assert normalize_names(["  EXAMPLE  ", " "]) == ["example"]


**Q2 · Keep one category · Easy**

Return only rows whose category equals the requested category.


In [ ]:
def select_category(rows, category):
    return [row for row in rows if row["category"] == category]

assert select_category([{"category": "overall"}, {"category": "coding"}], "coding") == [{"category": "coding"}]


**Q3 · Apply a vote minimum · Easy**

Keep rows with a known vote_count at least min_votes.


In [ ]:
def with_min_votes(rows, min_votes):
    return [row for row in rows if row["vote_count"] is not None and row["vote_count"] >= min_votes]

assert with_min_votes([{"vote_count": None}, {"vote_count": 0}], 0) == [{"vote_count": 0}]


**Q4 · Find requested models · Easy**

Keep rows matching any name in an already normalized name list.


In [ ]:
def find_models(rows, names):
    return [row for row in rows if row["model_name"].casefold() in names]

assert find_models([{"model_name": "Example"}], ["example"]) == [{"model_name": "Example"}]


**Q5 · Order known ratings · Medium**

Sort known ratings highest first; break ties by lowercase model name.


In [ ]:
def sort_by_rating(rows):
    known = [row for row in rows if row["rating"] is not None]
    return sorted(known, key=lambda row: (-row["rating"], row["model_name"].casefold()))

assert sort_by_rating([{"rating": None, "model_name": "missing"}]) == []


**Q6 · Limit the shortlist · Easy**

Return the first limit rows of an already sorted list.


In [ ]:
def take_top(rows, limit):
    return rows[:limit]

assert take_top(["a", "b", "c"], 2) == ["a", "b"]


**Q7 · Measure interval width · Easy**

Return rating_upper minus rating_lower, or None if either is missing.


In [ ]:
def interval_width(row):
    low, high = row["rating_lower"], row["rating_upper"]
    return None if low is None or high is None else high - low

assert interval_width({"rating_lower": 2, "rating_upper": 5}) == 3


**Q8 · Check interval overlap · Medium**

Return whether two closed intervals overlap; return None for missing bounds.


In [ ]:
def intervals_overlap(a, b):
    bounds = (a["rating_lower"], a["rating_upper"], b["rating_lower"], b["rating_upper"])
    if any(value is None for value in bounds):
        return None
    return max(bounds[0], bounds[2]) <= min(bounds[1], bounds[3])

assert intervals_overlap({"rating_lower": 0, "rating_upper": 1}, {"rating_lower": 1, "rating_upper": 2}) is True


**Q9 · Validate lookup arguments · Medium**

Accept only category, min_votes, and limit with valid types and ranges.


In [ ]:
def valid_lookup_args(args, categories):
    if not isinstance(args, dict) or set(args) != {"category", "min_votes", "limit"}:
        return False
    if not isinstance(args["category"], str):
        return False
    if type(args["min_votes"]) is not int or type(args["limit"]) is not int:
        return False
    return args["category"] in categories and args["min_votes"] >= 0 and 1 <= args["limit"] <= 5

assert valid_lookup_args({"category": "overall", "min_votes": 0, "limit": 3}, ["overall"]) is True


**Q10 · Attach source evidence · Easy**

Return source, category, retrieved_at, published_at, and models in one dictionary.


In [ ]:
def make_tool_result(rows, category, retrieved_at, published_at):
    return {"source": ARENA_SOURCE, "category": category, "retrieved_at": retrieved_at,
            "published_at": published_at, "models": rows}

assert make_tool_result([], "overall", "now", "publication")["source"] == ARENA_SOURCE


## 3. Read current evidence

Arena ratings summarize human preference in a particular category. They are not dollar prices, speed measurements, or a guarantee for your business task. Confidence bounds and vote counts help us discuss uncertainty. Overlapping intervals do not establish a statistical significance result.

The supplied source adapter handles HTTP and pagination. Focus on the short functions above and how their results enter the tool trace.


In [ ]:
def fetch_arena(category):
    if category not in ARENA_CATEGORIES:
        raise ValueError("Choose a category from ARENA_CATEGORIES.")
    try:
        import pyarrow.parquet as parquet
    except ImportError:
        raise RuntimeError("Install pyarrow once with %pip install -q pyarrow, then rerun.") from None
    from io import BytesIO
    retrieved_at = utc_now()
    # Resolve the current revision on EVERY run, then read its latest split.
    # The revision is chosen now, never hard-coded to an old release.
    repo = "lmarena-ai/leaderboard-dataset"
    revision = get_json("https://huggingface.co/api/datasets/" + repo)["sha"]
    files = get_json(f"https://huggingface.co/api/datasets/{repo}/tree/{revision}/text_style_control")
    paths = sorted(entry["path"] for entry in files if entry["type"] == "file"
                   and entry["path"].startswith("text_style_control/latest-")
                   and entry["path"].endswith(".parquet"))
    if not paths:
        raise RuntimeError("The latest split moved; inspect the source before continuing.")
    rows = []
    for path in paths:
        raw = get_bytes(f"https://huggingface.co/datasets/{repo}/resolve/{revision}/{path}")
        rows.extend(parquet.read_table(BytesIO(raw)).to_pylist())
    rows = select_category(rows, category)
    if not rows:
        raise RuntimeError("No current rows for this category; check the source menu.")
    dates = sorted({row["leaderboard_publish_date"] for row in rows})
    if len(dates) != 1:
        raise RuntimeError("Publication changed during retrieval; refresh the whole run.")
    return {"rows": rows, "category": category, "retrieved_at": retrieved_at,
            "published_at": dates[0], "source_revision": revision}

def lookup_arena(snapshot, category, min_votes, limit):
    args = {"category": category, "min_votes": min_votes, "limit": limit}
    if not valid_lookup_args(args, ARENA_CATEGORIES):
        raise ValueError("Invalid lookup arguments.")
    if category != snapshot["category"]:
        raise ValueError("Load this category before using it; do not mix sources.")
    rows = select_category(snapshot["rows"], category)
    rows = take_top(sort_by_rating(with_min_votes(rows, min_votes)), limit)
    fields = ("model_name", "rating", "rating_lower", "rating_upper", "vote_count", "category")
    evidence = [{**{field: row[field] for field in fields}, "interval_width": interval_width(row)} for row in rows]
    return make_tool_result(evidence, category, snapshot["retrieved_at"], snapshot["published_at"])


In [ ]:
# Run this cell in class. Re-running it fetches the source's latest release.
live_arena = fetch_arena("overall")
evidence = lookup_arena(live_arena, "overall", min_votes=0, limit=3)
show(evidence)

# Practice an exact lookup with three names taken from this live result.
requested = normalize_names([row["model_name"] for row in evidence["models"]])
selected = find_models(live_arena["rows"], requested)
print("Exact matches:", [row["model_name"] for row in selected])
if len(evidence["models"]) >= 2:
    print("First two intervals overlap:", intervals_overlap(*evidence["models"][:2]))

# Two valid requests and one invalid request, using the same current run.
for args in [{"category": "overall", "min_votes": 0, "limit": 3},
             {"category": "overall", "min_votes": 1000, "limit": 2},
             {"category": "unsupported", "min_votes": 0, "limit": 3}]:
    try:
        result = lookup_arena(live_arena, **args)
        print(args, "->", len(result["models"]), "models")
    except ValueError as error:
        print(args, "-> rejected:", str(error))


## 4. Let the model propose actions

The model requests one tool call. Python validates its arguments, runs the tool, and passes the evidence back for an answer. Inspect the trace before judging the answer.

The controller cell defines the workflow without making a paid request. Set RUN_LLM to True after setup, then rerun it. The baseline remains useful before the controller is enabled.


In [ ]:
# The instructor supplies a tool-capable controller model from the course gateway.
# Set LLM_MODEL and LLM_API_KEY in the environment; do not paste keys into cells.
# LLM_BASE_URL defaults to the OpenAI API, but can be an approved compatible gateway.
def ask_controller(messages, tools, tool_choice="auto"):
    model = os.getenv("LLM_MODEL", "")
    key = os.getenv("LLM_API_KEY", "")
    if not model or not key:
        raise RuntimeError("Configure LLM_MODEL and LLM_API_KEY before running the controller.")
    base = os.getenv("LLM_BASE_URL", "https://api.openai.com/v1").rstrip("/")
    body = {"model": model, "messages": messages, "tools": tools,
            "tool_choice": tool_choice, "parallel_tool_calls": False,
            "max_completion_tokens": 1500}
    request = Request(base + "/chat/completions", data=json.dumps(body).encode(),
                      headers={"Authorization": "Bearer " + key,
                               "Content-Type": "application/json"}, method="POST")
    try:
        with urlopen(request, timeout=60) as response:
            result = json.load(response)
    except HTTPError as error:
        raise RuntimeError(f"Controller HTTP {error.code}; check gateway, model access, or quota.") from None
    except (URLError, TimeoutError):
        raise RuntimeError("Controller unavailable; check course gateway access.") from None
    return result["choices"][0]["message"]

def tool_schema(name, description, properties):
    return {"type": "function", "function": {"name": name, "description": description,
            "parameters": {"type": "object", "properties": properties,
                           "required": list(properties), "additionalProperties": False}}}

def one_tool_workflow(question, snapshot):
    schema = tool_schema("lookup_arena", "Read current Arena ratings in the loaded category.", {
        "category": {"type": "string", "enum": [snapshot["category"]]},
        "min_votes": {"type": "integer", "minimum": 0},
        "limit": {"type": "integer", "minimum": 1, "maximum": 5}})
    messages = [{"role": "system", "content":
        "Use exactly one lookup. Arena measures aggregate human preference, not cost or latency. "
        "After the result, cite its source and publication date; explain uncertainty. "
        "Do not invent models, scores, votes, or statistical significance."},
        {"role": "user", "content": question}]
    proposal = ask_controller(messages, [schema], {"type": "function", "function": {"name": "lookup_arena"}})
    calls = proposal.get("tool_calls") or []
    if len(calls) != 1 or calls[0]["function"]["name"] != "lookup_arena":
        raise ValueError("Expected one allowed tool call.")
    call = calls[0]
    try:
        args = json.loads(call["function"]["arguments"])
    except (ValueError, TypeError):
        raise ValueError("Tool arguments were not valid JSON.") from None
    if not valid_lookup_args(args, ARENA_CATEGORIES):
        raise ValueError("Tool arguments failed validation.")
    observation = lookup_arena(snapshot, **args)
    messages += [{"role": "assistant", "content": proposal.get("content"), "tool_calls": calls},
                 {"role": "tool", "tool_call_id": call["id"], "content": json.dumps(observation)}]
    answer = ask_controller(messages, [schema], "none")
    return {"trace": [{"tool": "lookup_arena", "arguments": args, "observation": observation}],
            "answer": answer.get("content", "")}

RUN_LLM = False  # Set True after configuring the course controller environment.
if RUN_LLM:
    run = one_tool_workflow("Shortlist three current overall models. Explain what the ratings cannot tell us.", live_arena)
    show(run["trace"])
    print(run["answer"])


## 5. Discuss and complete the activity

Explain one rejected call, one uncertainty in the top-three shortlist, and one decision that these ratings cannot settle. Check every name, number, and citation in the controller's answer against its observation.

Record the source and retrieval time, leaderboard publication date. Refresh for a new class run; today’s winner is not an answer key.

Source: [Arena aggregate leaderboard dataset](https://huggingface.co/datasets/lmarena-ai/leaderboard-dataset). Arena dataset attribution: lmarena-ai, CC BY 4.0. Controller protocol: [official function-calling documentation](https://developers.openai.com/api/docs/guides/function-calling).
